# Former-reservoir stations: phase 0 audit

**Question:** could the analysis add stations that sat **in the former reservoir**? Those stations change from still water to river when the dam is removed, which the literature suggests is where water quality changes most. Today every upstream station flagged as possibly in the former reservoir is **dropped** (`exclude_former_reservoir` in `analysis/mixed_model.py`); the distance models do the same.

**Read-only.** This notebook changes nothing: no pipeline step, classification rule or model. It answers six questions so the design can be decided on numbers:
1. How many dams have stations in the former reservoir, under **each** way of identifying them (and their combinations)?
2. How many of those have enough measurements before and after removal, under the main rule (3 samples in 2 years per side) and the looser one (3 in 1 year)?
3. How many measurements have usable **depth**, and how many dams keep enough data if only surface samples are used? Did the sampling organization or activity type change at removal?
4. How many dams also have a usable **control station above** the former reservoir?
5. Among the upstream stations used as controls **today**, how many have an **unknown** reservoir status (or are flagged by another signal)?
6. Which measures have enough dams for the possible designs?

**Three signals, reported separately.** None is treated as the truth, and they aren't independent or equally reliable:
- **length:** the station is within the rough reservoir length (dam height ÷ river slope) upstream of the dam (`upstream_in_former_reservoir`; unknown when height or slope is missing). A crude wedge estimate.
- **polygon:** the dam sits at the outlet of a mapped NHD waterbody (Stage 4 `reservoir_status = dam_at_outlet`) and the station's own flowline runs through **that** waterbody (`WBArea_Permanent_Identifier` = the dam's reservoir). Unknown for dams without a mapped polygon. A second version uses only polygons mapped by the removal year.
- **waterbody flowline:** the station's reach is an NHD artificial path (FCode 55800), i.e. a line through some lake, reservoir or wide river; it doesn't prove the station was in *this* reservoir.

**Possible designs (for context, not decided):** *reservoir before/after* (change at reservoir stations; descriptive: it can't separate the removal from other changes over time) and *reservoir BACI* (reservoir stations vs controls above the reservoir; with downstream stations too, the three-zone version). **Dam counts:** the same rules as the rest of the project: at least **10** dams to fit, and **15+** before a result loses the *exploratory* label. Counts here come before the models' own filters (perennial streams, covariates, exposure), so the models would have fewer.

In [ ]:
from pathlib import Path
import os
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from analysis import mixed_model as mm
from etl.common.config import load_config
from etl.common.reservoir import AT_OUTLET, PRE_REMOVAL, polygon_timing, read_dam_reaches
from etl.common.warehouse import analysis_path, warehouse_dir
from etl.common.water_features import clean_permanent_id

DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))
CASE_STUDY = "all_removals"
cfg = load_config(CASE_STUDY, project_root=DATA_ROOT)
WAREHOUSE = warehouse_dir(cfg)
PROCESSED = cfg.project_root / "data" / "processed"
MIN_FIT, NOT_EXPLORATORY = 10, 15     # the project's existing dam-count rules
SURFACE_M = 1.0                       # "surface sample": depth <= 1 m (screening value)
SERIES = mm.SETTINGS["series"]
CONTROL_EXPOSURE = mm.SETTINGS["control_exposure"]
TABLES = {"3 / 2 (main)": (analysis_path(WAREHOUSE), 2), "3 / 1": (analysis_path(WAREHOUSE, "min1yr"), 1)}

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
print(f"Warehouse: {WAREHOUSE}")

## 1. Upstream station–dam pairs and the three signals

Every linked upstream pair (within the 10 km upstream limit), whatever its data. Each signal is true, false or unknown.

In [ ]:
with duckdb.connect(str(WAREHOUSE / "warehouse.duckdb"), read_only=True) as con:
    up = con.sql("""
        SELECT AR_ID, station_id, network_distance_km, backwater_km,
               upstream_in_former_reservoir AS sig_length, station_reach_fcode
        FROM link_features WHERE side = 'upstream'
    """).df()
up["AR_ID"] = up["AR_ID"].astype(str)

dams = pd.read_parquet(WAREHOUSE / "dam_removal.parquet", columns=["AR_ID", "Year_Removed"])
dams["AR_ID"] = dams["AR_ID"].astype(str)
reaches = read_dam_reaches(PROCESSED / "dam_reaches" / CASE_STUDY)
res = dams.merge(reaches, on="AR_ID", how="left")
res["at_outlet"] = res["reservoir_status"].eq(AT_OUTLET).fillna(False).astype(bool)
res["timing"] = (polygon_timing(res["reservoir_fdate"], res["Year_Removed"]).where(res["at_outlet"])
                 if "reservoir_fdate" in res else pd.NA)
res["reservoir_id"] = clean_permanent_id(res["reservoir_id"])

stations = pd.read_parquet(WAREHOUSE / "wq_station.parquet", columns=["station_id", "NHDPlusID", "huc8"])
pairs = (up.merge(res[["AR_ID", "at_outlet", "reservoir_id", "timing"]], on="AR_ID", how="left")
           .merge(stations, on="station_id", how="left"))
print(f"Upstream pairs: {len(pairs):,} ({pairs['AR_ID'].nunique()} dams, {pairs['station_id'].nunique()} stations); "
      f"of them at dams with a mapped reservoir polygon: {int(pairs['at_outlet'].sum()):,}")

In [ ]:
# Polygon signal: the WBArea_Permanent_Identifier of each station's flowline, read from its HUC8 package
# (attributes only, no geometry), for upstream pairs at dams with a mapped polygon.
import pyogrio
from etl.common.nhdplus import find_huc8_geodatabase

need = pairs[pairs["at_outlet"] & pairs["reservoir_id"].notna() & pairs["NHDPlusID"].notna()]
wbarea, missing_packages = {}, set()
for huc8, group in need.groupby("huc8"):
    try:
        path = find_huc8_geodatabase(PROCESSED / "nhdplus_extracted", huc8)
    except FileNotFoundError:
        missing_packages.add(huc8); continue
    fields = {f.lower(): f for f in pyogrio.read_info(path, layer="NHDFlowline")["fields"]}
    id_field, wb_field = fields.get("nhdplusid"), fields.get("wbarea_permanent_identifier")
    if not id_field or not wb_field:
        missing_packages.add(huc8); continue
    ids = sorted({int(x) for x in group["NHDPlusID"]})
    for i in range(0, len(ids), 500):
        part = ids[i:i + 500]
        t = pyogrio.read_dataframe(path, layer="NHDFlowline", columns=[id_field, wb_field], read_geometry=False,
                                   where=f"{id_field} IN ({','.join(map(str, part))})")
        wbarea.update({int(a): b for a, b in zip(t[id_field], t[wb_field]) if pd.notna(a)})
print(f"Station flowlines looked up: {len(wbarea):,}; HUC8 packages missing: {sorted(missing_packages) or 'none'}")

pairs["station_wbarea"] = clean_permanent_id(pairs["NHDPlusID"].map(lambda x: wbarea.get(int(x)) if pd.notna(x) else None))
looked_up = pairs["at_outlet"] & pairs["NHDPlusID"].map(lambda x: pd.notna(x) and int(x) in wbarea)
pairs["sig_polygon"] = (pairs["station_wbarea"].eq(pairs["reservoir_id"]).fillna(False)).where(looked_up).astype("boolean")
pairs["sig_polygon_pre"] = (pairs["sig_polygon"] & pairs["timing"].eq(PRE_REMOVAL).fillna(False)) \
    .where(looked_up).astype("boolean")
pairs["sig_waterbody_path"] = pairs["station_reach_fcode"].eq(55800).where(pairs["station_reach_fcode"].notna()).astype("boolean")
pairs["sig_length"] = pairs["sig_length"].astype("boolean")

SIGNALS = {"length": "sig_length", "polygon": "sig_polygon", "polygon (mapped by removal year)": "sig_polygon_pre",
           "waterbody flowline": "sig_waterbody_path"}
rows = []
for name, col in SIGNALS.items():
    s = pairs[col]
    rows.append({"signal": name, "pairs_true": int(s.eq(True).sum()), "pairs_false": int(s.eq(False).sum()),
                 "pairs_unknown": int(s.isna().sum()), "stations_true": pairs.loc[s.eq(True), "station_id"].nunique(),
                 "dams_true": pairs.loc[s.eq(True), "AR_ID"].nunique()})
print("Q1a. Each signal on its own (all linked upstream pairs):")
display(pd.DataFrame(rows))

In [ ]:
# Q1b. Combinations: how the signals agree. T = true, F = false, ? = unknown.
code3 = lambda s: s.map({True: "T", False: "F"}).fillna("?")
combo = pairs.assign(length=code3(pairs["sig_length"]), polygon=code3(pairs["sig_polygon"]),
                     waterbody_flowline=code3(pairs["sig_waterbody_path"]))
table = (combo.groupby(["length", "polygon", "waterbody_flowline"])
         .agg(pairs=("station_id", "size"), stations=("station_id", "nunique"), dams=("AR_ID", "nunique"))
         .reset_index().sort_values("pairs", ascending=False))
print("Q1b. Signal combinations (all linked upstream pairs):")
display(table)

# Candidate definitions, for counting only; none is chosen here.
t = lambda c: pairs[c].fillna(False).astype(bool)
n_true = t("sig_length").astype(int) + t("sig_polygon").astype(int) + t("sig_waterbody_path").astype(int)
DEFINITIONS = {
    "length": t("sig_length"),
    "polygon": t("sig_polygon"),
    "polygon (mapped by removal year)": t("sig_polygon_pre"),
    "waterbody flowline": t("sig_waterbody_path"),
    "length or polygon": t("sig_length") | t("sig_polygon"),
    "length and polygon": t("sig_length") & t("sig_polygon"),
    ">= 2 signals": n_true >= 2,
    "any signal": n_true >= 1,
}
for name, mask in DEFINITIONS.items():
    pairs[f"def::{name}"] = mask
pairs["any_signal_true"] = n_true >= 1
print("Q1c. Dams with at least one station in the former reservoir, per definition (all linked pairs):")
display(pd.DataFrame([{"definition": k, "stations": pairs.loc[m, "station_id"].nunique(), "dams": pairs.loc[m, "AR_ID"].nunique()}
                      for k, m in DEFINITIONS.items()]))

## 2. Enough data before and after removal?

The analysis tables hold only pairs that meet the eligibility rule (per dam × station × measure). Here: the upstream rows of each table, restricted to the model measures and to stations not exposed to another removal (`upstream_control`, as for controls today).

In [ ]:
DEF_COLS = [c for c in pairs.columns if c.startswith("def::")]
SIG_COLS = list(SIGNALS.values())
cols = ["measurement_id", "AR_ID", "station_id", "side", "period", "year", "parameter_std", "sample_fraction_std",
        "network_distance_km", "exposure_status", "upstream_in_former_reservoir"]
data = {}
for label, (path, min_years) in TABLES.items():
    if not path.exists():
        print(f"{label}: {path.name} missing; skipped"); continue
    a = pd.read_parquet(path, columns=cols)
    a["AR_ID"] = a["AR_ID"].astype(str)
    a["series"] = mm.series_name(a)
    a = a[a["series"].isin(SERIES)]
    data[label] = (a, min_years)

def upstream_rows(a):
    u = a[(a["side"] == "upstream") & a["exposure_status"].isin(CONTROL_EXPOSURE)]
    return u.merge(pairs[["AR_ID", "station_id"] + SIG_COLS + DEF_COLS + ["any_signal_true"]],
                   on=["AR_ID", "station_id"], how="left")

def dams_with(u, mask):
    return u.loc[mask, "AR_ID"].nunique()

rows = []
for label, (a, _) in data.items():
    u = upstream_rows(a)
    for c in DEF_COLS:
        m = u[c].fillna(False).astype(bool)
        rows.append({"eligibility": label, "definition": c[5:], "dams": dams_with(u, m),
                     "stations": u.loc[m, "station_id"].nunique(), "rows": int(m.sum())})
print("Q2. Dams with an eligible former-reservoir station (any model measure):")
display(pd.DataFrame(rows).pivot_table(index="definition", columns="eligibility", values=["dams", "stations", "rows"])
        .reindex([c[5:] for c in DEF_COLS]))

## 3. Sample depth, and did sampling change at removal?

Before removal a reservoir station may have been sampled as a lake (sometimes at depth); after, as a river at the surface. **First:** how many reservoir-station measurements have a depth at all, and how deep. **Then:** dams still eligible when (a) every sample is kept, (b) samples with known depth > 1 m are dropped (unknown depth kept), (c) only samples with known depth ≤ 1 m are kept. Eligibility is recomputed with the pipeline's rule (≥ 3 samples and ≥ 2 years, or ≥ 1 year for 3 / 1, per side); it ignores the single-year month rule, so it's approximate. **Last:** whether the organization or activity type differs before vs after at the same station.

In [ ]:
def eligible(rows, min_years):
    g = rows.groupby(["AR_ID", "station_id", "series", "period"]).agg(n=("measurement_id", "size"), years=("year", "nunique"))
    ok = ((g["n"] >= 3) & (g["years"] >= min_years)).unstack("period").reindex(columns=["pre", "post"]).fillna(False)
    return ok[ok.all(axis=1)].reset_index()[["AR_ID", "station_id", "series"]]

wq_path = WAREHOUSE / "wq_measurement.parquet"
depth_rows, change_rows, elig_rows = [], [], []
res_rows = {}
for label, (a, min_years) in data.items():
    u = upstream_rows(a)
    r = u[u["any_signal_true"].fillna(False).astype(bool)]
    ids = pd.DataFrame({"measurement_id": r["measurement_id"].unique()})
    con = duckdb.connect()
    con.register("ids", ids)
    extra = con.sql(f"""SELECT measurement_id, depth_m, organization_id, activity_type
                        FROM read_parquet('{wq_path}') JOIN ids USING (measurement_id)""").df()
    con.close()
    r = r.merge(extra, on="measurement_id", how="left")
    res_rows[label] = r
    known = r["depth_m"].notna()
    band = pd.cut(r["depth_m"], [-np.inf, SURFACE_M, 3, np.inf], labels=[f"<= {SURFACE_M:g} m", "1-3 m", "> 3 m"])
    d = (r.assign(depth=band.astype("string").fillna("unknown")).groupby(["period", "depth"]).size()
         .unstack("depth", fill_value=0))
    d.insert(0, "eligibility", label)
    depth_rows.append(d.reset_index())
    for c in DEF_COLS:
        rr = r[r[c].fillna(False).astype(bool)]
        for subset, keep in (("all samples", pd.Series(True, index=rr.index)),
                             ("drop known depth > 1 m", ~(rr["depth_m"] > SURFACE_M)),
                             ("known depth <= 1 m only", rr["depth_m"] <= SURFACE_M)):
            e = eligible(rr[keep.fillna(False).astype(bool)], min_years)
            elig_rows.append({"eligibility": label, "definition": c[5:], "samples": subset, "dams": e["AR_ID"].nunique()})
    by_pair = r.groupby(["AR_ID", "station_id", "period"]).agg(orgs=("organization_id", lambda s: frozenset(s.dropna())),
                                                              acts=("activity_type", lambda s: frozenset(s.dropna())))
    w = by_pair.unstack("period").dropna()
    change_rows.append({"eligibility": label, "pairs": len(w),
                        "organization differs pre vs post": int((w[("orgs", "pre")] != w[("orgs", "post")]).sum()),
                        "activity type differs pre vs post": int((w[("acts", "pre")] != w[("acts", "post")]).sum())})

print("Q3a. Depth of measurements at former-reservoir stations (any signal), by period:")
display(pd.concat(depth_rows, ignore_index=True))
print("Q3b. Dams still eligible under each depth subset:")
display(pd.DataFrame(elig_rows).pivot_table(index=["definition", "samples"], columns="eligibility", values="dams")
        .reindex([c[5:] for c in DEF_COLS], level=0))
print("Q3c. Station-dam pairs where the sampling organization or activity type differs before vs after:")
display(pd.DataFrame(change_rows))

## 4. Control stations above the former reservoir

For reservoir BACI a dam also needs an upstream station that is **not** in the former reservoir and lies **farther upstream** than its reservoir stations, eligible for the same measure. Two versions: *strict* (no signal true and the length signal known to be false) and *relaxed* (no signal true; unknown allowed). The three-zone version also needs an eligible downstream station within 10 km (treated, as in the existing models).

In [ ]:
def zone_counts(a, min_years, definition):
    u = upstream_rows(a)
    is_res = u[definition].fillna(False).astype(bool)
    res_ = u[is_res]
    top = res_.groupby("AR_ID")["network_distance_km"].max().rename("reservoir_top_km")
    u = u.merge(top, on="AR_ID", how="left")
    no_signal = ~u["any_signal_true"].fillna(False).astype(bool)
    above = u["network_distance_km"] > u["reservoir_top_km"]
    strict = no_signal & u["sig_length"].eq(False).fillna(False).astype(bool) & above
    relaxed = no_signal & above
    down = a[(a["side"] == "downstream") & a["exposure_status"].isin(mm.SETTINGS["treated_exposure"])
             & (a["network_distance_km"] <= 10)]
    keys = lambda f: set(map(tuple, f[["AR_ID", "series"]].drop_duplicates().to_numpy()))
    r_k, s_k, x_k, d_k = keys(res_), keys(u[strict]), keys(u[relaxed]), keys(down)
    out = []
    for series in SERIES:
        pick = lambda k: {d for d, s in k if s == series}
        r, s, x, d = pick(r_k), pick(s_k), pick(x_k), pick(d_k)
        out.append({"series": series, "reservoir_before_after": len(r),
                    "reservoir_BACI_strict": len(r & s), "reservoir_BACI_relaxed": len(r & x),
                    "three_zone_strict": len(r & s & d), "three_zone_relaxed": len(r & x & d)})
    return pd.DataFrame(out)

FOCUS = ["def::length", "def::polygon", "def::length or polygon", "def::any signal"]
q4 = []
for label, (a, min_years) in data.items():
    for definition in FOCUS:
        z = zone_counts(a, min_years, definition)
        q4.append(z.assign(eligibility=label, definition=definition[5:]))
q4 = pd.concat(q4, ignore_index=True)
print("Q4. Dams per measure: reservoir stations alone, with a control above, and with downstream too:")
display(q4.set_index(["eligibility", "definition", "series"]))

## 5. Today's controls: how many have an unknown reservoir status?

The models now treat an upstream station as a control unless the length signal is **true**; unknown counts as a control. Here: today's control pairs, split by the length signal, and how many of them another signal places in a waterbody or the dam's reservoir.

In [ ]:
for label, (a, _) in data.items():
    u = upstream_rows(a)
    controls = u[~u["upstream_in_former_reservoir"].fillna(False).astype(bool)]
    by = controls.drop_duplicates(["AR_ID", "station_id"])
    status = by["sig_length"].map({False: "length: false (outside)", True: "length: true"}).fillna("length: unknown")
    summary = (by.assign(status=status).groupby("status")
               .agg(pairs=("station_id", "size"), stations=("station_id", "nunique"), dams=("AR_ID", "nunique")))
    print(f"\nQ5. {label}: upstream control pairs used today, by length signal")
    display(summary)
    other = by[by["sig_polygon"].eq(True) | by["sig_waterbody_path"].eq(True)]
    print(f"  of these, polygon or waterbody-flowline signal true: {len(other)} pairs at {other['AR_ID'].nunique()} dams "
          f"(polygon: {int(by['sig_polygon'].eq(True).sum())}, waterbody flowline: {int(by['sig_waterbody_path'].eq(True).sum())})")
    unk = by[by["sig_length"].isna()]
    baci_dams = a[a["side"] == "downstream"]["AR_ID"].unique()
    print(f"  dams whose controls are ALL unknown-status: "
          f"{int((by.groupby('AR_ID')['sig_length'].apply(lambda s: s.isna().all())).sum())} "
          f"(of {by['AR_ID'].nunique()} dams with controls; {len(set(unk['AR_ID']) & set(baci_dams))} of them also have downstream data)")

## 6. Which measures have enough dams?

From section 4, per measure: the number of dams for each possible design, marked against the project's rules (**fit** ≥ 10 dams; **not exploratory** ≥ 15). Shown for every definition counted in section 4, with all samples; section 3 shows how much a surface-only rule would cost. These counts come **before** the models' own filters, so treat a measure just above 10 as unlikely to survive them.

In [ ]:
def mark(n):
    return f"{n} (fit, not exploratory)" if n >= NOT_EXPLORATORY else (f"{n} (fit, exploratory)" if n >= MIN_FIT else str(n))

designs = ["reservoir_before_after", "reservoir_BACI_strict", "reservoir_BACI_relaxed", "three_zone_relaxed"]
view = q4.copy()
for d in designs:
    view[d] = view[d].map(mark)
for label in data:
    print(f"\n{label}:")
    display(view[view["eligibility"] == label].pivot_table(index="series", columns="definition", values=designs,
                                                          aggfunc="first").reindex(SERIES))
counts = q4.melt(id_vars=["eligibility", "definition", "series"], value_vars=designs, var_name="design", value_name="dams")
print("\nMeasures reaching >= 10 dams, per design / definition / eligibility:")
display(counts[counts["dams"] >= MIN_FIT].groupby(["eligibility", "definition", "design"])["series"]
        .apply(lambda s: ", ".join(s)).unstack("design").fillna(""))

**Reading it (to decide after seeing the numbers, not here):** which definition to use for the main analysis and which for sensitivity; whether a surface-only rule is affordable; whether reservoir BACI (with a control above) has enough dams for any measure or only the descriptive reservoir before/after does; and whether the unknown-status controls in section 5 call for a sensitivity run of the **existing** BACI models without them. Nothing here changes the pipeline or the models.